In [1]:
import os
from pathlib import Path
import pandas as pd
import wget
import matplotlib.pyplot as plt
import numpy as np

from atomworks.io import parse
from atomworks.io.utils.testing import get_pdb_path_or_buffer
from atomworks.io.utils.visualize import view


Environment variable CCD_MIRROR_PATH not set. Will not be able to use function requiring this variable. To set it you may:
  (1) add the line 'export VAR_NAME=path/to/variable' to your .bashrc or .zshrc file
  (2) set it in your current shell with 'export VAR_NAME=path/to/variable'
  (3) write it to a .env file in the root of the atomworks.io repository


In [4]:
df = pd.read_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate.csv")
stoichiometry_df = pd.read_csv(Path.home() / "data/bad_afdb/stoichiometry_results_full.csv")
print(df.columns)
print(stoichiometry_df.columns)
df = pd.merge(df, stoichiometry_df, on="pdb", how="left")
print(f"Full dataset: {len(df)}")
df = df[df["stoichiometry"] == "A1"]
print(f"After stoichiometry filter: {len(df)}")

df = df[df["length"] >= 50]
print(f"After length >= 50 filter: {len(df)}")

df = df[df["length"] <= 640]
print(f"After length <= 640 filter: {len(df)}")

df.to_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_length_50-640.csv", index=False)

# df = df[df["plddt"] <= 50]
# print(f"After pLDDT <= 50 filter: {len(df)}")

# df.to_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_plddt_50_length_50-640.csv", index=False)

# plt.hist(df["plddt"])
# plt.axvline(x=50, color="red", linestyle="--", label=f"pLDDT <= 50, 50-640 length ({len(filtered_df)}/{len(df)})")
# plt.legend()
# plt.show()


Index(['pdb', 'uniprot', 'identity', 'plddt', 'coverage', 'length'], dtype='object')
Index(['pdb', 'stoichiometry', 'oligomeric_state', 'full_composition'], dtype='object')
Full dataset: 25871
After stoichiometry filter: 5269
After length >= 50 filter: 5001
After length <= 640 filter: 4663


In [2]:
import subprocess
from tqdm import tqdm

df = pd.read_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_length_50-640.csv")

# Directories
pdb_dir = Path.home() / "data/bad_afdb/pdb"
afdb_dir = Path.home() / "data/bad_afdb/afdb"
afdb_dir.mkdir(parents=True, exist_ok=True)

# Results storage
results = []

# Process each row
for index, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    pdb_chain_id = row["pdb"]
    uniprot_id = row["uniprot"]
    pdb_id, chain_id = pdb_chain_id.split("_")
    
    # PDB file path (middle two chars as subdir)
    pdb_subdir = pdb_dir / pdb_id[1:3]
    pdb_file = pdb_subdir / f"{pdb_id}.cif"
    
    # Download PDB if needed
    if not pdb_file.exists():
        pdb_subdir.mkdir(parents=True, exist_ok=True)
        subprocess.run(["wget", "-q", f"https://files.rcsb.org/download/{pdb_id}.cif", "-O", str(pdb_file)])
    
    # AFDB file path (using v6 - current version)
    afdb_file = afdb_dir / f"AF-{uniprot_id}-F1-model_v6.cif"
    
    # Download AFDB if needed
    if not afdb_file.exists():
        url = f"https://alphafold.ebi.ac.uk/files/AF-{uniprot_id}-F1-model_v6.cif"
        result = subprocess.run(["wget", "-q", url, "-O", str(afdb_file)], capture_output=True)
        if result.returncode != 0:
            print(f"Failed to download AFDB for {uniprot_id}")
            results.append({
                "pdb_chain": pdb_chain_id,
                "uniprot": uniprot_id,
                "tm_score": None,
                "error": "AFDB download failed"
            })
            continue
    
    # Check files exist
    if not pdb_file.exists():
        results.append({
            "pdb_chain": pdb_chain_id,
            "uniprot": uniprot_id,
            "tm_score": None,
            "error": "PDB file missing"
        })
        continue
    
    if not afdb_file.exists() or afdb_file.stat().st_size == 0:
        results.append({
            "pdb_chain": pdb_chain_id,
            "uniprot": uniprot_id,
            "tm_score": None,
            "error": "AFDB file missing or empty"
        })
        continue
    
    # Run USalign: PDB chain vs AFDB (TM normalized by PDB chain length = TM1)
    # USalign <pdb> -chain1 <chain> <afdb> -outfmt 2
    cmd = [
        "USalign",
        str(pdb_file),
        "-chain1", chain_id,
        str(afdb_file),
        "-outfmt", "2",
        "-TMscore", "5",
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode != 0:
        results.append({
            "pdb_chain": pdb_chain_id,
            "uniprot": uniprot_id,
            "tm_score": None,
            "error": f"USalign failed: {result.stderr[:100]}"
        })
        continue
    
    # Parse USalign output for TM1 (normalized by first structure = PDB chain)
    tm_score = None
    for line in result.stdout.strip().split("\n"):
        if line.startswith("#") or not line.strip():
            continue
        parts = line.split("\t")
        if len(parts) >= 3:
            tm_score = float(parts[2])  # TM1 column
            break
    
    results.append({
        "pdb_chain": pdb_chain_id,
        "uniprot": uniprot_id,
        "tm_score": tm_score,
        "error": None
    })

# Create results dataframe
results_df = pd.DataFrame(results)
results_df.to_csv(Path.home() / "data/bad_afdb/pdb_vs_afdb_tm_scores.csv", index=False)

# Summary
print(f"Total: {len(results_df)}")
print(f"Successful: {results_df['tm_score'].notna().sum()}")
print(f"Failed: {results_df['tm_score'].isna().sum()}")
print(f"\nTM-score statistics:")
print(results_df['tm_score'].describe())


Processing:   0%|          | 0/4663 [00:00<?, ?it/s]

Processing: 100%|██████████| 4663/4663 [06:04<00:00, 12.80it/s]

Total: 4663
Successful: 4545
Failed: 118

TM-score statistics:
count    4545.000000
mean        0.915732
std         0.133555
min         0.021000
25%         0.917500
50%         0.966900
75%         0.986200
max         0.999700
Name: tm_score, dtype: float64


In [3]:
results_df.head()

,pdb_chain,uniprot,tm_score,error
0,5QBC_A,P11838,0.9989,None
1,5QCS_B,P56817,0.9916,None
2,5QIX_A,Q96DC9,0.9798,None
3,5QKU_A,P0AEG4,0.9662,None
4,5QTY_A,P03951,0.9827,None


In [4]:
df_tm = df.merge(results_df[["uniprot", "tm_score"]], on="uniprot", how="left")
df_tm.head()
df_tm.to_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_length_50-640_tm.csv", index=False)


In [5]:
df_tm = pd.read_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_length_50-640_tm.csv")
df_tm = df_tm[df_tm["tm_score"] < 0.5]
print(len(df_tm))
print(df_tm["length"].describe())
print(df_tm.head())
df_tm.to_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_length_50-640_tm-05.csv", index=False)


157
count    157.000000
mean     157.503185
std      111.463782
min       50.000000
25%       77.000000
50%      121.000000
75%      206.000000
max      635.000000
Name: length, dtype: float64
        pdb uniprot  identity      plddt  coverage  length stoichiometry  \
278  6SLO_A  Q9UHX1  0.504673  61.749206  0.382826     214            A1   
287  6SUS_A  P0DKX7  1.000000  90.267403  0.151231     258            A1   
313  6TPU_A  P10586  0.994845  86.438041  0.101730     194            A1   
334  6UF2_A  Q31PX7  1.000000  93.797903  0.992000     124            A1   
387  6WPC_A  Q6S554  0.083916  95.106713  0.729592     143            A1   

    oligomeric_state full_composition  tm_score  
278          Monomer           ['A1']    0.4777  
287          Monomer           ['A1']    0.2877  
313          Monomer           ['A1']    0.4548  
334          Monomer           ['A1']    0.4655  
387          Monomer           ['A1']    0.3315  


In [8]:
df_tm = pd.read_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_length_50-640_tm-05.csv")
df_tm = df_tm[df_tm["coverage"] > 0.8]
df_tm = df_tm[df_tm["identity"] >= 0.7]
print(len(df_tm))
print(df_tm["length"].describe())
print(df_tm.head())
df_tm.to_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_monomer_length_50-640_tm-05_coverage-08_identity-07.csv", index=False)


36
count     36.000000
mean     205.527778
std      122.334316
min       62.000000
25%      117.750000
50%      159.500000
75%      290.000000
max      635.000000
Name: length, dtype: float64
       pdb uniprot  identity      plddt  coverage  length stoichiometry  \
3   6UF2_A  Q31PX7  1.000000  93.797903  0.992000     124            A1   
7   6ZVY_A  P0A3G3  0.917241  98.524966  0.989761     290            A1   
8   7A2D_A  P64596  0.994152  92.843684  0.895288     171            A1   
14  7ZIX_A  P0A3G3  0.924138  98.524966  0.989761     290            A1   
17  8AUC_B  Q8NQA0  1.000000  80.120142  0.820000     492            A1   

   oligomeric_state full_composition  tm_score  
3           Monomer           ['A1']    0.4655  
7           Monomer           ['A1']    0.4520  
8           Monomer           ['A1']    0.4592  
14          Monomer           ['A1']    0.4520  
17          Monomer           ['A1']    0.3043  


In [12]:
df = pd.read_csv(Path.home() / "data/bad_afdb/pdb_70_cluster_reps_aligned_confidence_aggregate_plddt_50_length_50-768.csv")
pdb_dir = Path.home() / "data/bad_afdb/pdb"
pdb_dir.mkdir(parents=True, exist_ok=True)

for index, row in df.iterrows():
    pdb_chain_id = row["pdb"]
    pdb_id, chain_id = pdb_chain_id.split("_")
    pdb_subdir = pdb_dir / pdb_id[1:3]
    pdb_subdir.mkdir(parents=True, exist_ok=True)
    pdb_file = pdb_subdir / f"{pdb_id}.cif"
    if not pdb_file.exists():
        wget.download(f"https://files.rcsb.org/download/{pdb_id}.cif", str(pdb_file))
    # result_dict = parse(
    #     filename=get_pdb_path_or_buffer(pdb_id),
    #     build_assembly=["1"],
    #     add_missing_atoms=True,
    #     remove_waters=True,
    #     hydrogen_policy="remove",
    #     model=1,
    # )
    # print("Keys in parsed result:", list(result_dict.keys()))
    # asym_unit = result_dict["asym_unit"][0]
    # asym_unit = asym_unit[asym_unit.occupancy > 0]
    # view(asym_unit)
    # assembly = result_dict["assemblies"]["1"][0]
    # assembly = assembly[assembly.occupancy > 0]
    # view(assembly)

    # print("Chain info:", result_dict["chain_info"])
    # print("Ligand info:", result_dict["ligand_info"])
    # print("Metadata:", result_dict["metadata"])
    # print("Annotation categories:", result_dict["asym_unit"][0].get_annotation_categories())

    # for chain_id, info in result_dict["chain_info"].items():
        # print(info.keys())
        # print(chain_id, info["processed_entity_canonical_sequence"])


In [7]:
cross_protein_summary_path = Path.home() / "proteina/inference/inference_seq_cond_sampling_finetune-all_8-seq_purge-7bny-7kww-7ad5-7b76_045-noise/bad_afdb_cross_protein_analysis/cross_protein_summary_data.csv"
inference_data_path = Path.home() / "proteina/inference/inference_seq_cond_sampling_finetune-all_8-seq_purge-7bny-7kww-7ad5-7b76_045-noise/"
cross_protein_summary_data = pd.read_csv(cross_protein_summary_path)
print(cross_protein_summary_data.columns)
cross_protein_summary_data["tm_gain"] = cross_protein_summary_data["tm_ref_pred_at_max_ptm"] - cross_protein_summary_data["tms_msa"]
cross_protein_summary_data.sort_values(by="tm_gain", ascending=False, inplace=True)
print(cross_protein_summary_data[cross_protein_summary_data["in_train"] == False].head(10))


Index(['protein_id', 'spearman_rho_composite', 'spearman_rho_ptm',
       'max_tm_ref_template', 'max_tm_ref_pred',
       'tm_ref_template_at_max_composite', 'tm_ref_pred_at_max_composite',
       'tm_ref_pred_at_max_ptm', 'top_1_tm_ref_template',
       'top_5_tm_ref_template', 'top_1_tm_ref_pred', 'top_5_tm_ref_pred',
       'tms_msa', 'in_train', 'length'],
      dtype='object')
    protein_id  spearman_rho_composite  spearman_rho_ptm  max_tm_ref_template  \
92      7ZK0_A                0.741674          0.637971              0.68552   
81      7ZKD_A                0.189965          0.286865              0.52094   
19      6TF4_A                0.403211          0.244504              0.65449   
136     8ZWZ_A                0.460202          0.531158              0.75073   
33      8JB9_A                0.466143          0.448983              0.62478   
50     7N6G_1Q                0.344378          0.420013              0.46978   
11      8HTU_K                0.496953         

In [9]:
cross_protein_summary_data.sort_values(by="tm_ref_pred_at_max_ptm", ascending=True, inplace=True)
print(cross_protein_summary_data.head(10))

    protein_id  spearman_rho_composite  spearman_rho_ptm  max_tm_ref_template  \
78     8GLV_7I                0.116014         -0.000292              0.12802   
39     8GLV_7P               -0.156837         -0.122844              0.18302   
79     9E2G_3T               -0.003219          0.002931              0.15069   
29     9E78_2O                0.061748          0.057508              0.18393   
41     8I7O_E2                0.053322         -0.045574              0.19555   
109    8GZU_a7               -0.082717         -0.461497              0.18836   
53      7KWZ_A               -0.062996         -0.143254              0.19512   
103    8GZU_0E               -0.028757         -0.154505              0.18743   
116     8B12_X                0.152397          0.257148              0.21421   
63      8YVE_X                0.175820          0.101330              0.20455   

     max_tm_ref_pred  tm_ref_template_at_max_composite  \
78           0.12711                           0.0

In [2]:
pdb_file = str(Path.home() / "data/cath_v4-4/by_fold/3.30.70/5k8mA02")
result_dict = parse(filename=pdb_file, file_type="pdb", build_assembly=["1"], add_missing_atoms=True, remove_waters=True, hydrogen_policy="remove", model=1)
print(f">5k8mA02\n{result_dict['chain_info']['A']['processed_entity_canonical_sequence']}")

pdb_file = str(Path.home() / "data/7ad5_example/7ad5.pdb")
result_dict = parse(filename=pdb_file, file_type="pdb", build_assembly=["1"], add_missing_atoms=True, remove_waters=True, hydrogen_policy="remove", model=1)
print(f">7ad5\n{result_dict['chain_info']['A']['processed_entity_canonical_sequence']}")


/home/jupyter-chenxi/.conda/envs/atomworks/lib/python3.12/site-packages/biotite/structure/io/pdb/file.py:470: UserWarning: 1035 elements were guessed from atom name
  warnings.warn(
Chain A contains both polymer and non-polymer residues; separating them for processing, naming the non-polymer residues as B.
Chain A contains both polymer and non-polymer residues; separating them for processing, naming the non-polymer residues as B.


>5k8mA02
EAGKVFLKVTVFAGPSRYTPYVPRPVAALDSPNAIVRAKLVEALEEWADNYEKRYTREYGGGTVVPKVAIGAIRGGVPYKIYRFPELCSIYDIRLNPDTNPLVVQREVEAVVSKLGLKAEVKPFLFRRG


No suitable coordinates found for 'NI' among preferences ('ideal_pdbx', 'model', 'ideal_rdkit'). Coordinates will be 'nan'.


>7ad5
GHMHDCHQVTVSRDVTLQNKERHDCNQVCASIDKETENKLNTDIIPRLTRYMSVKGNSIIARVQQSNSDPKCSCTWRAIIWRVYKAYDENSLNVALHVSHPNQQIGENPDWSLVISNPNVHCLK
